# MNIST Handwritten Digit Classification — ANN and CNN

This notebook trains and evaluates both a fully connected ANN and a CNN on MNIST. It includes preprocessing, visualisation, parameter-count calculations, metrics, confusion matrices, prediction examples, and model saving.

## Dataset links
- [Official Keras MNIST loader](https://keras.io/api/datasets/mnist/) — downloads 60,000 training and 10,000 test images automatically.
- [Kaggle MNIST CSV dataset](https://www.kaggle.com/datasets/whenamancodes/mnist-easy) — `mnist_train.csv` and `mnist_test.csv`, with one label column and 784 pixel columns.
- [Kaggle Digit Recognizer competition](https://www.kaggle.com/competitions/digit-recognizer/data) — CSV format, useful if you want to practise Kaggle workflows.
- [GitHub MNIST files](https://github.com/fgnt/mnist) — raw IDX gzip files.

The notebook uses the Keras loader by default, so no manual dataset download is required.

In [ ]:
# Install packages if required
# Run this cell once in a terminal or uncomment it in Jupyter:
# %pip install tensorflow scikit-learn pandas numpy matplotlib seaborn

# Optional Kaggle CSV download (requires Kaggle API credentials):
# !kaggle datasets download -d whenamancodes/mnist-easy -p data --unzip

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization,
    Conv2D, MaxPooling2D, Flatten
)
from tensorflow.keras.optimizers import Adam

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
print("TensorFlow:", tf.__version__)

## 1. Load MNIST

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)
print("Pixel range:", X_train.min(), "to", X_train.max())
print("Classes:", np.unique(y_train))

## 2. Visualise sample images

In [ ]:
plt.figure(figsize=(10, 5))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(f"Label = {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 3. Normalize and prepare labels

In [ ]:
X_train_norm = X_train.astype("float32") / 255.0
X_test_norm = X_test.astype("float32") / 255.0

num_classes = 10
y_train_onehot = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_onehot = tf.keras.utils.to_categorical(y_test, num_classes)

# ANN inputs: flatten each 28x28 image to 784 features
X_train_flat = X_train_norm.reshape(-1, 784)
X_test_flat = X_test_norm.reshape(-1, 784)

# CNN inputs: retain spatial structure and add grayscale channel
X_train_cnn = X_train_norm[..., np.newaxis]
X_test_cnn = X_test_norm[..., np.newaxis]

print("ANN input:", X_train_flat.shape)
print("CNN input:", X_train_cnn.shape)
print("Example one-hot label:", y_train_onehot[0])

## 4. Train-validation splits

In [ ]:
X_train_ann, X_val_ann, y_train_ann, y_val_ann = train_test_split(
    X_train_flat, y_train_onehot, test_size=0.10,
    random_state=RANDOM_STATE, stratify=y_train
)

X_train_cnn, X_val_cnn, y_train_cnn, y_val_cnn = train_test_split(
    X_train_cnn, y_train_onehot, test_size=0.10,
    random_state=RANDOM_STATE, stratify=y_train
)

## 5. ANN model

In [ ]:
ann_model = Sequential([
    Input(shape=(784,)),
    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.30),
    Dense(64, activation="relu"),
    Dropout(0.20),
    Dense(10, activation="softmax")
])
ann_model.compile(optimizer=Adam(0.001), loss="categorical_crossentropy", metrics=["accuracy"])
ann_model.summary()

### ANN parameter formulas
For a Dense layer, parameters = `(input features × units) + units` for the bias. Dropout has zero parameters. BatchNormalization has trainable scale and offset parameters plus non-trainable moving statistics.

In [ ]:
def parameter_table(model):
    rows = []
    for layer in model.layers:
        rows.append({
            "Layer": layer.name,
            "Type": layer.__class__.__name__,
            "Output shape": str(layer.output.shape),
            "Parameters": layer.count_params()
        })
    return pd.DataFrame(rows)

parameter_table(ann_model)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

history_ann = ann_model.fit(
    X_train_ann, y_train_ann,
    validation_data=(X_val_ann, y_val_ann),
    epochs=15, batch_size=128,
    callbacks=[early_stop], verbose=1
)

## 6. CNN model

In [ ]:
cnn_model = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.50),
    Dense(10, activation="softmax")
])
cnn_model.compile(optimizer=Adam(0.001), loss="categorical_crossentropy", metrics=["accuracy"])
cnn_model.summary()

### CNN parameter calculations

For Conv2D: parameters = `(kernel height × kernel width × input channels + 1) × number of filters`.

- Conv2D-1: `(3 × 3 × 1 + 1) × 32 = 320`.
- Conv2D-2: `(3 × 3 × 32 + 1) × 64 = 18,496`.
- After valid 3×3 convolutions and 2×2 pooling, the feature map is 7×7×64, so Flatten produces 3,136 values.
- Dense(128): `3,136 × 128 + 128 = 401,536`.
- Output Dense(10): `128 × 10 + 10 = 1,290`.

The following cells calculate and verify these values automatically.

In [ ]:
parameter_table(cnn_model)

In [ ]:
cnn_model.summary()

In [ ]:
history_cnn = cnn_model.fit(
    X_train_cnn, y_train_cnn,
    validation_data=(X_val_cnn, y_val_cnn),
    epochs=15, batch_size=128,
    callbacks=[early_stop], verbose=1
)

## 7. Training curves

In [ ]:
def plot_history(history, title):
    h = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(h["loss"], label="Train")
    axes[0].plot(h["val_loss"], label="Validation")
    axes[0].set_title(f"{title} Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[1].plot(h["accuracy"], label="Train")
    axes[1].plot(h["val_accuracy"], label="Validation")
    axes[1].set_title(f"{title} Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(history_ann, "ANN")
plot_history(history_cnn, "CNN")

## 8. Evaluation metrics

In [ ]:
def evaluate_classifier(model, X, y_onehot, y_true, name, image=False):
    loss, accuracy = model.evaluate(X, y_onehot, verbose=0)
    probabilities = model.predict(X, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    scores = {
        "Model": name,
        "Loss": loss,
        "Accuracy": accuracy,
        "Precision": precision_score(y_true, predictions, average="weighted", zero_division=0),
        "Recall": recall_score(y_true, predictions, average="weighted", zero_division=0),
        "F1": f1_score(y_true, predictions, average="weighted", zero_division=0)
    }
    print(f"\n{name} classification report")
    print(classification_report(y_true, predictions, digits=4))
    return scores, predictions, probabilities

ann_scores, y_pred_ann, y_prob_ann = evaluate_classifier(
    ann_model, X_test_flat, y_test_onehot, y_test, "ANN"
)
cnn_scores, y_pred_cnn, y_prob_cnn = evaluate_classifier(
    cnn_model, X_test_cnn, y_test_onehot, y_test, "CNN"
)

comparison = pd.DataFrame([ann_scores, cnn_scores])
display(comparison.style.format({
    "Loss": "{:.4f}", "Accuracy": "{:.4f}",
    "Precision": "{:.4f}", "Recall": "{:.4f}", "F1": "{:.4f}"
}))

## 9. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(confusion_matrix(y_test, y_pred_ann), annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("ANN Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(confusion_matrix(y_test, y_pred_cnn), annot=True, fmt="d", cmap="Greens", ax=axes[1])
axes[1].set_title("CNN Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()

## 10. Prediction example

In [ ]:
index = 100

plt.figure(figsize=(4, 4))
plt.imshow(X_test[index], cmap="gray")
plt.axis("off")
plt.title(f"Actual digit: {y_test[index]}")
plt.show()

ann_prob = ann_model.predict(X_test_flat[index:index+1], verbose=0)[0]
cnn_prob = cnn_model.predict(X_test_cnn[index:index+1], verbose=0)[0]

print("ANN prediction:", np.argmax(ann_prob), f"confidence={ann_prob.max():.2%}")
print("CNN prediction:", np.argmax(cnn_prob), f"confidence={cnn_prob.max():.2%}")
print("CNN probabilities:", np.round(cnn_prob, 4))

## 11. Save models

In [ ]:
ann_model.save("mnist_ann_model.keras")
cnn_model.save("mnist_cnn_model.keras")
print("Saved mnist_ann_model.keras and mnist_cnn_model.keras")

## 12. Optional: load Kaggle CSV files

If you downloaded the CSV version from Kaggle, place the files in a `data/` folder and run the following. Kaggle's CSV layout has one label column followed by 784 pixel columns.

In [ ]:
# Optional CSV workflow
# train_df = pd.read_csv("data/mnist_train.csv")
# test_df = pd.read_csv("data/mnist_test.csv")
#
# y_train_csv = train_df.iloc[:, 0].to_numpy()
# X_train_csv = train_df.iloc[:, 1:].to_numpy().reshape(-1, 28, 28).astype("float32") / 255.0
#
# y_test_csv = test_df.iloc[:, 0].to_numpy()
# X_test_csv = test_df.iloc[:, 1:].to_numpy().reshape(-1, 28, 28).astype("float32") / 255.0
#
# print(X_train_csv.shape, y_train_csv.shape)
# print(X_test_csv.shape, y_test_csv.shape)

## Key takeaways

- ANN input is flattened to 784 features.
- CNN input remains 28×28×1.
- Convolutional parameters use shared filters, while Dense parameters connect every input to every neuron.
- `model.summary()` and `parameter_table()` show trainable parameter counts.
- CNNs usually perform better on image data because they preserve spatial information.